# KIPRIS CSV(XLSX) dataset to PDF (For RAG)

In [ ]:
# 1. 코랩 리눅스 환경에 나눔 폰트 설치 (한글 깨짐 방지 필수)
!apt-get install -y fonts-nanum

# 2. PDF 생성 및 데이터 핸들링 라이브러리 설치
!pip install fpdf2 pandas

# 3. CSV, XLSX 파일형식 무시하는 최신 모듈 설치
!pip install python-calamine

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
import pandas as pd
from fpdf import FPDF
import os

# =========================================================================
# [설정] 업로드한 KIPRIS 데이터 파일 이름을 적어주세요.
# =========================================================================
file_name = 'drive/MyDrive/comento/2w/prototype_dataset.xlsx'
df = pd.read_excel(file_name,engine='calamine')

In [ ]:
# -------------------------------------------------------------------------
# 1. 한글 전용 PDF 포맷 레이아웃 정의
# -------------------------------------------------------------------------
class KEA_Patent_PDF(FPDF):
    def header(self):
        self.set_font('NanumGothic', '', 9)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, 'KEA Tech-GPT RAG 시스템 참조 데이터셋 (KIPRIS 외부 연계 인덱스)', 0, 1, 'L')
        self.line(10, 17, 200, 17)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font('NanumGothic', '', 8)
        self.set_text_color(128, 128, 128)
        self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')

pdf = KEA_Patent_PDF()

# 시스템에 설치된 나눔고딕 폰트 경로 등록
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
pdf.add_font('NanumGothic', '', font_path)
pdf.set_font('NanumGothic', '', 10)
pdf.set_auto_page_break(auto=True, margin=15)

In [ ]:
# -------------------------------------------------------------------------
# 2. 엑셀/CSV 행(Row)을 순회하며 고밀도 자연어 텍스트 조립 후 PDF 적재
# -------------------------------------------------------------------------
# 다운로드받은 KIPRIS 컬럼명과 아래 대괄호 안의 이름이 일치하는지 확인해 주세요.
for idx, row in df.iterrows():
    pdf.add_page()

    # 결측치(Empty cell) 대비 안전하게 문자열 변환
    reg_num = str(row.get('등록번호', row.get('출원번호', '정보없음'))).strip()
    title = str(row.get('발명의명칭', row.get('발명의 명칭', '정보없음'))).strip()
    applicant = str(row.get('출원인', row.get('최종권리자', '정보없음'))).strip()
    summary = str(row.get('요약', '정보없음')).strip()
    claims = str(row.get('청구항', '정보없음')).strip()

    # [A] 특허 헤더 매핑 (등록번호, 출원인 명시)
    pdf.set_font('NanumGothic', '', 12)
    pdf.set_text_color(0, 51, 102) # 진한 네이비색 포인트
    pdf.multi_cell(0, 8, f"문서 인덱스 No.{idx+1} : {title}")
    pdf.ln(2)

    # [B] 메타데이터 작성
    pdf.set_font('NanumGothic', '', 10)
    pdf.set_text_color(30, 30, 30)
    pdf.multi_cell(0, 6, f"■ 관리 번호 : KIPRIS-REG-{reg_num}\n■ 최종 특허권자(출원인) : {applicant}")
    pdf.line(10, pdf.get_y()+2, 200, pdf.get_y()+2)
    pdf.ln(5)

    # [C] 기술 요약 독해 영역
    pdf.set_font('NanumGothic', '', 11)
    pdf.set_text_color(102, 51, 0) # 브라운색 포인트
    pdf.cell(0, 6, "【발명의 요약】", 0, 1)
    pdf.set_font('NanumGothic', '', 9.5)
    pdf.set_text_color(50, 50, 50)
    pdf.multi_cell(0, 5.5, summary)
    pdf.ln(4)

    # [D] 법률 추론의 핵심인 청구범위 (Claims) 독해 영역
    pdf.set_font('NanumGothic', '', 11)
    pdf.set_text_color(204, 0, 0) # 붉은색 포인트 (중요 정보 강조)
    pdf.cell(0, 6, "【특허 청구범위 및 권리 범위 명세】", 0, 1)
    pdf.set_font('NanumGothic', '', 9.5)
    pdf.set_text_color(50, 50, 50)
    pdf.multi_cell(0, 5.5, claims)

/tmp/ipykernel_5624/3897776516.py:8: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, 'KEA Tech-GPT RAG 시스템 참조 데이터셋 (KIPRIS 외부 연계 인덱스)', 0, 1, 'L')
/tmp/ipykernel_5624/3466344391.py:31: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 6, "【발명의 요약】", 0, 1)
/tmp/ipykernel_5624/3466344391.py:40: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 6, "【특허 청구범위 및 권리 범위 명세】", 0, 1)
/tmp/ipykernel_5624/3897776516.py:16: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')


In [ ]:
# -------------------------------------------------------------------------
# 3. 최종 고밀도 텍스트 PDF 출력 및 저장
# -------------------------------------------------------------------------
output_filename = 'prototype_dataset.pdf'
pdf.output(output_filename)

print(f"==========================================")
print(f"생성된 파일: {os.path.abspath(output_filename)}")
print(f"==========================================")

/tmp/ipykernel_5624/3897776516.py:16: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=0 use new_x=XPos.RIGHT, new_y=YPos.TOP.
  self.cell(0, 10, f'Page {self.page_no()}', 0, 0, 'C')


✅ 변환 대성공!
✅ 생성된 파일: /content/prototype_dataset.pdf
